# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_09 — Volatility Surface Alpha Signals

### Research question

Can features extracted from an implied-volatility surface contain useful information about future realised volatility, volatility risk premium, and short-horizon returns — and how do we test those signals without overclaiming?

This notebook connects Phase 2 derivatives modelling with Phase 3 alpha research:

```text
02_09_volatility_surface_construction
02_11_heston_model_calibration
02_12_sabr_smile_calibration
02_14_local_volatility_dupire_surface
03_01_garch_volatility_modeling
03_02_har_realized_volatility_forecasting
```

The goal is not to claim that volatility surfaces automatically produce alpha.

The goal is to build a disciplined research pipeline for turning surface information into testable features.

It covers:

1. synthetic daily volatility-surface generation;
2. ATM volatility, skew, curvature, and term-structure features;
3. realised volatility and volatility risk premium;
4. surface feature z-scores;
5. forward targets: future realised variance, vol change, and returns;
6. information coefficient analysis;
7. decile/bucket diagnostics;
8. simple linear forecast models;
9. train/test split discipline;
10. toy volatility-risk-premium signal;
11. toy skew/term-structure signals;
12. transaction-cost and turnover diagnostics;
13. regime diagnostics;
14. limitations and research extensions.

The key idea:

> A volatility surface is not only a pricing object. It can be transformed into a structured feature set — but those features require careful out-of-sample validation before being called alpha.

## 1. What is a volatility surface feature?

A daily implied-volatility surface can be written as:

$$
\sigma_{\text{imp}}(k,T)
$$

where:

$$
k=\log(K/F_T)
$$

is log-forward-moneyness and $T$ is maturity.

Useful features include:

### ATM level

$$
\sigma_{\text{ATM}}(T) = \sigma_{\text{imp}}(0,T)
$$

### Skew

A simple skew proxy:

$$
skew(T) = \frac{ \sigma_{\text{imp}}(-k,T) - \sigma_{\text{imp}}(k,T) } {2k}
$$

### Curvature

A simple curvature proxy:

$$
\begin{aligned}
curvature(T) &= \frac{ \sigma_{\text{imp}}(-k,T) + \sigma_{\text{imp}}(k,T) - 2\sigma_{\text{imp}}(0,T) } {k^2}
\end{aligned}
$$

### Term structure slope

$$
\begin{aligned}
term\_slope &= \sigma_{\text{ATM}}(T_{\text{long}}) \\
&\quad - \sigma_{\text{ATM}}(T_{\text{short}})
\end{aligned}
$$

### Volatility risk premium

$$
\begin{aligned}
VRP_t &= \sigma_{\text{imp},t}^2 \\
&\quad - RV_{t,\text{past}}
\end{aligned}
$$

The idea is to ask whether these features forecast:

- future realised volatility;
- changes in implied volatility;
- risk-adjusted returns;
- option carry or hedging outcomes.

## 2. Why be careful?

Volatility-surface features are tempting.

But there are several traps:

1. **Look-ahead bias**  
   Using future realised volatility in today's signal.

2. **Overlapping labels**  
   Forward 21-day returns overlap heavily.

3. **Transaction costs**  
   Option spreads and hedging costs can destroy apparent alpha.

4. **Regime dependence**  
   Signals may work only in calm regimes.

5. **Surface construction bias**  
   Bad quote filtering or interpolation can create fake features.

6. **Carry versus crash risk**  
   Short-volatility signals can look great until stress events.

7. **Data-mining**  
   Many surface features create many chances for false discoveries.

This notebook treats surface features as research candidates, not as proof of tradable alpha.

## 3. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class SurfaceAlphaConfig:
    n_days: int = 2_500
    annualisation: int = 252
    train_fraction: float = 0.70
    lookahead_days: int = 21
    zscore_window: int = 252
    seed: int = 42


config = SurfaceAlphaConfig()
config

## 4. Synthetic market and surface data

We simulate a daily market with:

1. latent realised volatility;
2. implied volatility with a volatility risk premium;
3. equity-style skew that steepens in stress;
4. smile curvature;
5. term-structure dynamics;
6. returns that are weakly related to volatility regimes.

The synthetic data is designed to demonstrate the research pipeline, not to prove real-world profitability.

In [ ]:
def simulate_surface_market(config: SurfaceAlphaConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(config.seed)

    n = config.n_days
    dates = pd.bdate_range("2015-01-01", periods=n)

    latent_var = np.zeros(n)
    implied_atm_1m = np.zeros(n)
    implied_atm_3m = np.zeros(n)
    implied_atm_6m = np.zeros(n)
    skew_state = np.zeros(n)
    curvature_state = np.zeros(n)
    returns = np.zeros(n)

    long_run_var = (0.18 / np.sqrt(config.annualisation)) ** 2
    latent_var[0] = long_run_var
    skew_state[0] = 0.18
    curvature_state[0] = 0.12

    for t in range(1, n):
        vol_shock = rng.standard_normal()
        stress_jump = rng.uniform() < 0.012

        latent_var[t] = (
            0.985 * latent_var[t - 1]
            + 0.015 * long_run_var
            + 0.10 * latent_var[t - 1] * abs(vol_shock)
        )

        if stress_jump:
            latent_var[t] *= rng.uniform(1.8, 3.5)

        latent_var[t] = np.clip(latent_var[t], 1e-7, 0.003)

        latent_vol_ann = np.sqrt(latent_var[t] * config.annualisation)

        # Implied vols include risk premium and noise.
        vrp_level = 0.025 + 0.15 * latent_vol_ann + 0.02 * rng.standard_normal()
        implied_atm_1m[t] = np.clip(latent_vol_ann + vrp_level, 0.05, 1.50)

        term_slope = 0.015 + 0.10 * (0.20 - latent_vol_ann) + 0.01 * rng.standard_normal()
        implied_atm_3m[t] = np.clip(implied_atm_1m[t] + term_slope, 0.05, 1.50)
        implied_atm_6m[t] = np.clip(implied_atm_3m[t] + 0.5 * term_slope + 0.01 * rng.standard_normal(), 0.05, 1.50)

        # Skew steepens during high vol / negative returns.
        skew_state[t] = (
            0.95 * skew_state[t - 1]
            + 0.05 * (0.20 + 0.90 * latent_vol_ann)
            + 0.03 * rng.standard_normal()
        )
        skew_state[t] = np.clip(skew_state[t], 0.02, 1.50)

        curvature_state[t] = (
            0.94 * curvature_state[t - 1]
            + 0.06 * (0.10 + 0.45 * latent_vol_ann)
            + 0.02 * rng.standard_normal()
        )
        curvature_state[t] = np.clip(curvature_state[t], 0.01, 1.20)

        # Return process: weak risk premium, volatility drag, crash shocks.
        shock = rng.standard_t(df=7) * np.sqrt((7 - 2) / 7)
        drift = 0.00018 - 0.00045 * max(latent_vol_ann - 0.25, 0)
        returns[t] = drift + np.sqrt(latent_var[t]) * shock

        if stress_jump:
            returns[t] -= rng.uniform(0.015, 0.050)

    price = 100 * np.exp(np.cumsum(returns))

    market = pd.DataFrame({
        "date": dates,
        "return": returns,
        "price": price,
        "latent_realized_var_daily": latent_var,
        "latent_realized_vol_ann": np.sqrt(latent_var * config.annualisation),
        "atm_iv_1m": implied_atm_1m,
        "atm_iv_3m": implied_atm_3m,
        "atm_iv_6m": implied_atm_6m,
        "skew_state": skew_state,
        "curvature_state": curvature_state
    })

    # Build a full surface grid by date, maturity, and log-moneyness.
    maturities = np.array([21, 63, 126]) / config.annualisation
    maturity_labels = ["1m", "3m", "6m"]
    k_values = np.array([-0.30, -0.20, -0.10, 0.00, 0.10, 0.20, 0.30])

    rows = []

    for t in range(n):
        atm_by_label = {
            "1m": implied_atm_1m[t],
            "3m": implied_atm_3m[t],
            "6m": implied_atm_6m[t]
        }

        for T, label in zip(maturities, maturity_labels):
            atm = atm_by_label[label]
            maturity_scale = np.sqrt(21 / (T * config.annualisation))
            skew = skew_state[t] * maturity_scale
            curvature = curvature_state[t] * (0.80 + 0.20 * maturity_scale)

            for k in k_values:
                # Equity-style skew: downside k<0 vols higher.
                iv = atm - skew * k + curvature * k**2 + 0.004 * rng.standard_normal()
                iv = float(np.clip(iv, 0.03, 2.00))

                rows.append({
                    "date": dates[t],
                    "maturity_label": label,
                    "maturity_years": float(T),
                    "log_moneyness": float(k),
                    "implied_vol": iv
                })

    surface = pd.DataFrame(rows)

    return market, surface


market, surface = simulate_surface_market(config)

market.head(), surface.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(market["date"], market["price"])
plt.title("Synthetic Underlying Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(market["date"], market["latent_realized_vol_ann"], label="Latent realised vol")
plt.plot(market["date"], market["atm_iv_1m"], label="1m ATM IV", alpha=0.8)
plt.plot(market["date"], market["atm_iv_3m"], label="3m ATM IV", alpha=0.8)
plt.title("Realised Volatility and ATM Implied Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

## 5. Surface snapshot visualisation

We visualise one surface snapshot as volatility smiles by maturity.

The surface is indexed by:

$$
(k,T)
$$

where $k$ is log-forward-moneyness.

In [ ]:
snapshot_date = surface["date"].iloc[len(market) // 2]
snapshot = surface[surface["date"] == snapshot_date]

plt.figure(figsize=(10, 6))
for label, group in snapshot.groupby("maturity_label"):
    group = group.sort_values("log_moneyness")
    plt.plot(group["log_moneyness"], group["implied_vol"], marker="o", label=label)

plt.title(f"Implied Volatility Surface Snapshot: {snapshot_date.date()}")
plt.xlabel("Log-moneyness")
plt.ylabel("Implied volatility")
plt.legend()
plt.show()

## 6. Extracting surface features

For each date and maturity, we extract:

1. ATM volatility;
2. downside-upside skew;
3. curvature;
4. wing spread.

For a fixed $k_0=0.20$:

$$
skew(T) = \frac{\sigma(-k_0,T)-\sigma(k_0,T)} {2k_0}
$$

$$
curvature(T) = \frac{\sigma(-k_0,T)+\sigma(k_0,T)-2\sigma(0,T)} {k_0^2}
$$

Then across maturities:

$$
term\_slope = \sigma_{\text{ATM}}(6m)-\sigma_{\text{ATM}}(1m)
$$

$$
front\_skew = skew(1m)
$$

$$
skew\_term = skew(6m)-skew(1m)
$$

In [ ]:
def extract_surface_features(surface: pd.DataFrame, k0: float = 0.20) -> pd.DataFrame:
    rows = []

    for date, date_df in surface.groupby("date"):
        row = {"date": date}

        for label, g in date_df.groupby("maturity_label"):
            g = g.set_index("log_moneyness")["implied_vol"].sort_index()

            def get_iv(k):
                return float(g.loc[k])

            atm = get_iv(0.0)
            left = get_iv(-k0)
            right = get_iv(k0)
            far_left = get_iv(-0.30)
            far_right = get_iv(0.30)

            skew = (left - right) / (2 * k0)
            curvature = (left + right - 2 * atm) / (k0 ** 2)
            wing_spread = far_left - far_right

            row[f"atm_iv_{label}"] = atm
            row[f"skew_{label}"] = skew
            row[f"curvature_{label}"] = curvature
            row[f"wing_spread_{label}"] = wing_spread

        row["term_slope_6m_1m"] = row["atm_iv_6m"] - row["atm_iv_1m"]
        row["term_slope_3m_1m"] = row["atm_iv_3m"] - row["atm_iv_1m"]
        row["skew_term_6m_1m"] = row["skew_6m"] - row["skew_1m"]
        row["curvature_term_6m_1m"] = row["curvature_6m"] - row["curvature_1m"]

        rows.append(row)

    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


features = extract_surface_features(surface)

features.head()

## 7. Realised volatility and volatility risk premium

We compute past realised variance:

$$
RV_{t,21} = \sum_{i=0}^{20} r_{t-i}^2
$$

Annualised realised volatility:

$$
\sigma_{RV,t} = \sqrt{\frac{252}{21}RV_{t,21}}
$$

Variance risk premium proxy:

$$
\begin{aligned}
VRP_t &= \sigma_{\text{ATM},1m,t}^2 \\
&\quad - \sigma_{RV,t}^2
\end{aligned}
$$

This is a simplified feature.

A production variance risk premium calculation should use variance swap strikes or option replication, not just ATM IV.

In [ ]:
def add_realized_vol_and_targets(market: pd.DataFrame, features: pd.DataFrame, config: SurfaceAlphaConfig) -> pd.DataFrame:
    out = market.merge(features, on="date", how="inner").copy()

    h = config.lookahead_days

    out["rv_past_21d_var"] = out["return"].rolling(h).apply(lambda x: np.sum(x**2), raw=True)
    out["rv_past_21d_ann_vol"] = np.sqrt(config.annualisation / h * out["rv_past_21d_var"])

    out["vrp_1m_var"] = out["atm_iv_1m"]**2 - out["rv_past_21d_ann_vol"]**2
    out["vrp_3m_var"] = out["atm_iv_3m"]**2 - out["rv_past_21d_ann_vol"]**2

    # Forward targets.
    out["future_return_21d"] = np.log(out["price"].shift(-h) / out["price"])
    out["future_rv_21d_var"] = out["return"].shift(-1).rolling(h).apply(lambda x: np.sum(x**2), raw=True).shift(-(h-1))
    out["future_rv_21d_ann_vol"] = np.sqrt(config.annualisation / h * out["future_rv_21d_var"])

    out["future_realized_minus_atm_1m"] = out["future_rv_21d_ann_vol"]**2 - out["atm_iv_1m"]**2
    out["future_atm_iv_1m_change_21d"] = out["atm_iv_1m"].shift(-h) - out["atm_iv_1m"]
    out["future_skew_1m_change_21d"] = out["skew_1m"].shift(-h) - out["skew_1m"]

    # Binary stress target.
    out["future_stress_21d"] = (
        out["future_rv_21d_ann_vol"]
        > out["future_rv_21d_ann_vol"].rolling(252, min_periods=100).quantile(0.80)
    ).astype(float)

    out = out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    return out


research_df = add_realized_vol_and_targets(market, features, config)

research_df.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(research_df["date"], research_df["atm_iv_1m"], label="1m ATM IV")
plt.plot(research_df["date"], research_df["rv_past_21d_ann_vol"], label="Past 21d realised vol")
plt.plot(research_df["date"], research_df["future_rv_21d_ann_vol"], label="Future 21d realised vol", alpha=0.75)
plt.title("Implied Volatility vs Past/Future Realised Volatility")
plt.xlabel("Date")
plt.ylabel("Annualised volatility")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(research_df["date"], research_df["vrp_1m_var"])
plt.axhline(0, linestyle="--")
plt.title("Variance Risk Premium Proxy: 1m ATM IV² - Past RV²")
plt.xlabel("Date")
plt.ylabel("VRP proxy")
plt.show()

## 8. Rolling z-score features

Raw surface features can be non-stationary.

We create rolling z-scores:

$$
z_t(x)=\frac{x_t-\mu_{t,w}}{\sigma_{t,w}}
$$

where the rolling mean and standard deviation use past/current data only.

This produces features such as:

- unusually high VRP;
- unusually steep skew;
- unusually inverted term structure;
- unusually high curvature.

In [ ]:
surface_signal_cols = [
    "atm_iv_1m",
    "atm_iv_3m",
    "atm_iv_6m",
    "skew_1m",
    "skew_3m",
    "skew_6m",
    "curvature_1m",
    "curvature_3m",
    "curvature_6m",
    "wing_spread_1m",
    "term_slope_6m_1m",
    "term_slope_3m_1m",
    "skew_term_6m_1m",
    "curvature_term_6m_1m",
    "vrp_1m_var",
    "vrp_3m_var",
    "rv_past_21d_ann_vol"
]

def add_rolling_zscores(df: pd.DataFrame, cols: list[str], window: int) -> pd.DataFrame:
    out = df.copy()

    for col in cols:
        roll_mean = out[col].rolling(window, min_periods=max(50, window // 4)).mean()
        roll_std = out[col].rolling(window, min_periods=max(50, window // 4)).std()
        out[f"z_{col}"] = (out[col] - roll_mean) / roll_std.replace(0, np.nan)

    return out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)


research_df = add_rolling_zscores(research_df, surface_signal_cols, config.zscore_window)

z_feature_cols = [f"z_{c}" for c in surface_signal_cols]

research_df[["date"] + z_feature_cols[:6]].head()

## 9. Train/test split

We split chronologically.

All signal evaluation is reported on the test set unless explicitly labelled otherwise.

In [ ]:
split_idx = int(len(research_df) * config.train_fraction)

train = research_df.iloc[:split_idx].copy()
test = research_df.iloc[split_idx:].copy()

pd.Series({
    "n_total": len(research_df),
    "n_train": len(train),
    "n_test": len(test),
    "train_start": train["date"].iloc[0],
    "train_end": train["date"].iloc[-1],
    "test_start": test["date"].iloc[0],
    "test_end": test["date"].iloc[-1]
})

## 10. Information coefficient analysis

The information coefficient is:

$$
IC = \operatorname{corr}(signal_t, target_{t+h})
$$

We compute ICs for several targets:

1. future realised volatility;
2. future realised minus implied variance;
3. future 21-day return;
4. future ATM IV change.

This is signal research, not a trading backtest.

In [ ]:
target_cols = [
    "future_rv_21d_ann_vol",
    "future_realized_minus_atm_1m",
    "future_return_21d",
    "future_atm_iv_1m_change_21d",
    "future_skew_1m_change_21d"
]

def information_coefficient_table(df: pd.DataFrame, features: list[str], targets: list[str]) -> pd.DataFrame:
    rows = []

    for feature in features:
        for target in targets:
            x = df[feature]
            y = df[target]
            corr = x.corr(y)

            rows.append({
                "feature": feature,
                "target": target,
                "ic": corr,
                "abs_ic": abs(corr)
            })

    return pd.DataFrame(rows).sort_values("abs_ic", ascending=False)


ic_train = information_coefficient_table(train, z_feature_cols, target_cols)
ic_test = information_coefficient_table(test, z_feature_cols, target_cols)

ic_test.head(15)

In [ ]:
top_ic = ic_test.head(15).copy()
top_ic["label"] = top_ic["feature"] + "\n→ " + top_ic["target"]

plt.figure(figsize=(12, 7))
plt.barh(top_ic["label"][::-1], top_ic["ic"][::-1])
plt.axvline(0, linestyle="--")
plt.title("Top Test-Set Information Coefficients")
plt.xlabel("IC")
plt.ylabel("Feature → Target")
plt.show()

## 11. Bucket diagnostics

IC can hide nonlinear effects.

For a selected feature and target, we bucket feature values and inspect average future target.

A useful signal should have monotonic or economically interpretable bucket behaviour.

In [ ]:
def bucket_signal_diagnostic(df, feature, target, n_buckets=10):
    work = df[[feature, target]].dropna().copy()
    work["bucket"] = pd.qcut(work[feature], q=n_buckets, duplicates="drop")

    out = (
        work
        .groupby("bucket", observed=True)
        .agg(
            n=(target, "count"),
            mean_feature=(feature, "mean"),
            mean_target=(target, "mean"),
            median_target=(target, "median"),
            std_target=(target, "std")
        )
        .reset_index(drop=True)
    )

    out["target_standard_error"] = out["std_target"] / np.sqrt(out["n"])

    return out


selected_feature = "z_vrp_1m_var"
selected_target = "future_realized_minus_atm_1m"

bucket_vrp = bucket_signal_diagnostic(test, selected_feature, selected_target, n_buckets=10)

bucket_vrp

In [ ]:
plt.figure(figsize=(10, 6))
plt.errorbar(
    bucket_vrp["mean_feature"],
    bucket_vrp["mean_target"],
    yerr=1.96 * bucket_vrp["target_standard_error"],
    marker="o",
    capsize=4
)
plt.axhline(0, linestyle="--")
plt.title("Bucket Diagnostic: VRP z-score vs Future Realised Minus Implied Variance")
plt.xlabel("Mean feature value")
plt.ylabel("Mean future target")
plt.show()

## 12. Simple linear forecasting models

We fit lightweight linear models to forecast:

$$
future\_rv_{21d}
$$

and:

$$
future\_realised\_minus\_implied
$$

This is intentionally simple.

If a complex model is needed to find a tiny effect, the research should be treated cautiously.

In [ ]:
def fit_ols(X, y, ridge=1e-8):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    X_design = np.column_stack([np.ones(len(X)), X])

    xtx = X_design.T @ X_design
    xty = X_design.T @ y

    beta = np.linalg.solve(xtx + ridge * np.eye(xtx.shape[0]), xty)

    fitted = X_design @ beta
    residuals = y - fitted

    return {
        "beta": beta,
        "fitted": fitted,
        "residuals": residuals
    }


def predict_ols(model, X):
    X = np.asarray(X, dtype=float)
    X_design = np.column_stack([np.ones(len(X)), X])
    return X_design @ model["beta"]


model_feature_cols = [
    "z_atm_iv_1m",
    "z_skew_1m",
    "z_curvature_1m",
    "z_term_slope_6m_1m",
    "z_vrp_1m_var",
    "z_rv_past_21d_ann_vol"
]

models = {}
forecast_df = test[["date", "future_rv_21d_ann_vol", "future_realized_minus_atm_1m", "future_return_21d", "return"]].copy()

for target in ["future_rv_21d_ann_vol", "future_realized_minus_atm_1m"]:
    model = fit_ols(train[model_feature_cols].to_numpy(), train[target].to_numpy())
    models[target] = model

    forecast_df[f"forecast_{target}"] = predict_ols(model, test[model_feature_cols].to_numpy())

forecast_df.head()

## 13. Forecast metrics

We compare linear surface-feature forecasts against simple baselines:

- forecast future RV with today's 1m ATM IV;
- forecast future RV with past realised volatility;
- forecast realised-minus-implied with zero.

Metrics:

$$
MSE,\quad MAE,\quad correlation
$$

In [ ]:
def regression_metrics(y, pred, label):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    err = pred - y

    return {
        "forecast": label,
        "n": int(len(y)),
        "mse": float(np.mean(err**2)),
        "mae": float(np.mean(np.abs(err))),
        "corr": float(np.corrcoef(pred, y)[0, 1]) if np.std(pred) > 0 and np.std(y) > 0 else np.nan,
        "mean_prediction": float(np.mean(pred)),
        "mean_actual": float(np.mean(y))
    }


forecast_metrics = []

# Future RV target.
forecast_metrics.append(regression_metrics(
    test["future_rv_21d_ann_vol"],
    forecast_df["forecast_future_rv_21d_ann_vol"],
    "linear_surface_features_future_rv"
))

forecast_metrics.append(regression_metrics(
    test["future_rv_21d_ann_vol"],
    test["atm_iv_1m"],
    "baseline_atm_iv_1m_future_rv"
))

forecast_metrics.append(regression_metrics(
    test["future_rv_21d_ann_vol"],
    test["rv_past_21d_ann_vol"],
    "baseline_past_rv_future_rv"
))

# Realized minus implied target.
forecast_metrics.append(regression_metrics(
    test["future_realized_minus_atm_1m"],
    forecast_df["forecast_future_realized_minus_atm_1m"],
    "linear_surface_features_realized_minus_implied"
))

forecast_metrics.append(regression_metrics(
    test["future_realized_minus_atm_1m"],
    np.zeros(len(test)),
    "baseline_zero_realized_minus_implied"
))

forecast_metrics = pd.DataFrame(forecast_metrics).sort_values("mse")

forecast_metrics

## 14. Surface alpha signal definitions

We define simple candidate signals.

### 14.1 Volatility risk premium carry signal

If implied variance is high relative to past realised variance:

$$
signal^{VRP}_t=z(VRP_t)
$$

A short-volatility carry interpretation would take positive exposure when $VRP$ is high.

But this has crash risk.

### 14.2 Skew steepness signal

If skew is unusually steep:

$$
signal^{skew}_t=z(skew_t)
$$

This may proxy demand for downside protection.

### 14.3 Term-structure signal

If front volatility is high relative to back volatility:

$$
signal^{term}_t=-z(term\_slope_t)
$$

This may indicate stress or mean reversion in front volatility.

These are research signals, not production strategies.

In [ ]:
signal_df = test[[
    "date",
    "return",
    "future_return_21d",
    "future_rv_21d_ann_vol",
    "future_realized_minus_atm_1m",
    "future_atm_iv_1m_change_21d",
    "atm_iv_1m",
    "rv_past_21d_ann_vol",
    "latent_realized_vol_ann"
]].copy()

signal_df["signal_vrp_carry"] = test["z_vrp_1m_var"].clip(-3, 3)
signal_df["signal_skew_steepness"] = test["z_skew_1m"].clip(-3, 3)
signal_df["signal_term_stress"] = (-test["z_term_slope_6m_1m"]).clip(-3, 3)
signal_df["signal_surface_composite"] = (
    0.50 * signal_df["signal_vrp_carry"]
    + 0.25 * signal_df["signal_skew_steepness"]
    + 0.25 * signal_df["signal_term_stress"]
)

signal_cols = [
    "signal_vrp_carry",
    "signal_skew_steepness",
    "signal_term_stress",
    "signal_surface_composite"
]

signal_df[["date"] + signal_cols].head()

## 15. Signal IC and bucket tests

We test whether the candidate signals relate to:

1. future realised-minus-implied variance;
2. future realised volatility;
3. future underlying return;
4. future ATM IV change.

Again: signal association is not yet a tradeable strategy.

In [ ]:
signal_ic = information_coefficient_table(signal_df, signal_cols, [
    "future_realized_minus_atm_1m",
    "future_rv_21d_ann_vol",
    "future_return_21d",
    "future_atm_iv_1m_change_21d"
])

signal_ic.sort_values("abs_ic", ascending=False)

In [ ]:
bucket_composite = bucket_signal_diagnostic(
    signal_df,
    "signal_surface_composite",
    "future_realized_minus_atm_1m",
    n_buckets=10
)

plt.figure(figsize=(10, 6))
plt.errorbar(
    bucket_composite["mean_feature"],
    bucket_composite["mean_target"],
    yerr=1.96 * bucket_composite["target_standard_error"],
    marker="o",
    capsize=4
)
plt.axhline(0, linestyle="--")
plt.title("Composite Surface Signal Bucket Diagnostic")
plt.xlabel("Mean composite signal")
plt.ylabel("Future realised minus implied variance")
plt.show()

bucket_composite

## 16. Toy signal PnL

A real option strategy requires:

- option chain;
- bid/ask quotes;
- Greeks;
- delta hedging;
- margin;
- transaction costs;
- slippage;
- smile dynamics.

This notebook does **not** claim to simulate that fully.

Instead, we build a toy proxy:

$$
PnL_t^{VRP} = signal_t \cdot (\sigma_{\text{imp},t}^2 - RV_{t,t+h})
$$

This approximates the idea that a short-volatility signal benefits when implied variance exceeds future realised variance.

We include:

- position clipping;
- turnover cost;
- stress diagnostics.

This is a research diagnostic only.

In [ ]:
def toy_surface_signal_pnl(df, signal_col, cost_per_turnover=0.0005, max_abs_position=1.0):
    out = df[[
        "date",
        signal_col,
        "future_realized_minus_atm_1m",
        "future_return_21d",
        "future_rv_21d_ann_vol",
        "latent_realized_vol_ann"
    ]].copy()

    raw_signal = out[signal_col].to_numpy()
    position = np.clip(raw_signal / 2.0, -max_abs_position, max_abs_position)

    # Positive position means short variance-risk-premium exposure:
    # benefits when implied variance > future realised variance.
    # target = future_realized_minus_implied, so PnL is negative of target.
    gross = position * (-out["future_realized_minus_atm_1m"].to_numpy())

    turnover = np.abs(np.diff(np.r_[0.0, position]))
    cost = cost_per_turnover * turnover
    net = gross - cost

    out["position"] = position
    out["turnover"] = turnover
    out["gross_pnl_proxy"] = gross
    out["cost_proxy"] = cost
    out["net_pnl_proxy"] = net
    out["cum_net_pnl_proxy"] = np.cumsum(net)

    summary = {
        "signal": signal_col,
        "cost_per_turnover": cost_per_turnover,
        "mean_position": float(np.mean(position)),
        "mean_abs_position": float(np.mean(np.abs(position))),
        "mean_turnover": float(np.mean(turnover)),
        "total_gross_pnl_proxy": float(np.sum(gross)),
        "total_net_pnl_proxy": float(np.sum(net)),
        "mean_net_pnl_proxy": float(np.mean(net)),
        "vol_net_pnl_proxy": float(np.std(net, ddof=1)),
        "ir_proxy": float(np.mean(net) / np.std(net, ddof=1) * np.sqrt(252 / config.lookahead_days)) if np.std(net, ddof=1) > 0 else np.nan,
        "max_drawdown_proxy": float((np.cumsum(net) - np.maximum.accumulate(np.cumsum(net))).min())
    }

    return out, summary


toy_frames = []
toy_summaries = []

for sig in signal_cols:
    pnl_df, summary = toy_surface_signal_pnl(signal_df, sig, cost_per_turnover=0.0005)
    pnl_df["signal_name"] = sig
    toy_frames.append(pnl_df)
    toy_summaries.append(summary)

toy_pnl = pd.concat(toy_frames, ignore_index=True)
toy_summary = pd.DataFrame(toy_summaries).sort_values("ir_proxy", ascending=False)

toy_summary

In [ ]:
plt.figure(figsize=(12, 6))
for sig, group in toy_pnl.groupby("signal_name"):
    plt.plot(group["date"], group["cum_net_pnl_proxy"], label=sig)
plt.axhline(0, linestyle="--")
plt.title("Toy Surface Signal PnL Proxy")
plt.xlabel("Date")
plt.ylabel("Cumulative proxy PnL")
plt.legend()
plt.show()

## 17. Cost sensitivity

Any surface strategy can be killed by costs:

- option bid/ask;
- delta-hedging costs;
- slippage;
- margin;
- borrow/funding;
- implementation shortfall.

We vary the turnover cost and inspect whether toy signal performance survives.

In [ ]:
cost_grid = [0.0, 0.0001, 0.00025, 0.0005, 0.0010, 0.0020]
cost_rows = []

for cost in cost_grid:
    for sig in signal_cols:
        _, summary = toy_surface_signal_pnl(signal_df, sig, cost_per_turnover=cost)
        cost_rows.append(summary)

cost_sensitivity = pd.DataFrame(cost_rows)

cost_sensitivity.head()

In [ ]:
plt.figure(figsize=(10, 6))
for sig, group in cost_sensitivity.groupby("signal"):
    plt.plot(group["cost_per_turnover"], group["ir_proxy"], marker="o", label=sig)
plt.axhline(0, linestyle="--")
plt.title("Toy Signal IR Proxy vs Cost")
plt.xlabel("Cost per turnover")
plt.ylabel("IR proxy")
plt.legend()
plt.show()

## 18. Regime diagnostics

Volatility-surface signals often fail during stress.

We bucket by future realised volatility:

- low;
- medium;
- high;
- extreme.

Then evaluate toy PnL by regime.

A short-volatility carry signal can look strong on average but lose severely in extreme regimes.

In [ ]:
surface_pnl_best_signal = toy_summary.iloc[0]["signal"]
best_pnl_df = toy_pnl[toy_pnl["signal_name"] == surface_pnl_best_signal].copy()

best_pnl_df["future_vol_regime"] = pd.qcut(
    best_pnl_df["future_rv_21d_ann_vol"],
    q=4,
    labels=["low", "medium", "high", "extreme"]
)

regime_pnl = (
    best_pnl_df
    .groupby("future_vol_regime", observed=True)
    .agg(
        n=("net_pnl_proxy", "count"),
        mean_future_vol=("future_rv_21d_ann_vol", "mean"),
        mean_position=("position", "mean"),
        mean_net_pnl=("net_pnl_proxy", "mean"),
        total_net_pnl=("net_pnl_proxy", "sum"),
        pnl_vol=("net_pnl_proxy", "std"),
        worst_period=("net_pnl_proxy", "min")
    )
    .reset_index()
)

regime_pnl

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(regime_pnl["future_vol_regime"].astype(str), regime_pnl["mean_net_pnl"])
plt.axhline(0, linestyle="--")
plt.title(f"Mean Toy PnL by Future Vol Regime: {surface_pnl_best_signal}")
plt.xlabel("Future realised volatility regime")
plt.ylabel("Mean net PnL proxy")
plt.show()

## 19. Signal stability through time

A signal may work in one period and fail later.

We compute rolling IC for the composite signal.

This is important because volatility markets are regime-dependent.

In [ ]:
def rolling_ic(df, feature, target, window=252, step=21):
    rows = []

    for start in range(0, len(df) - window + 1, step):
        sub = df.iloc[start:start + window]
        rows.append({
            "start_date": sub["date"].iloc[0],
            "end_date": sub["date"].iloc[-1],
            "feature": feature,
            "target": target,
            "ic": sub[feature].corr(sub[target])
        })

    return pd.DataFrame(rows)


rolling_ic_df = rolling_ic(
    signal_df,
    "signal_surface_composite",
    "future_realized_minus_atm_1m",
    window=252,
    step=21
)

rolling_ic_df.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(rolling_ic_df["end_date"], rolling_ic_df["ic"], marker="o")
plt.axhline(0, linestyle="--")
plt.title("Rolling IC: Composite Surface Signal")
plt.xlabel("Window end")
plt.ylabel("Rolling IC")
plt.show()

## 20. Feature correlation and redundancy

Surface features are often highly correlated.

For example:

- ATM IV and skew often rise together in stress;
- curvature and wing spread may overlap;
- VRP and ATM IV can be mechanically related.

Highly correlated features can make alpha research unstable.

We inspect the feature correlation matrix.

In [ ]:
feature_corr = test[z_feature_cols].corr()

plt.figure(figsize=(10, 8))
plt.imshow(feature_corr.values, vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(z_feature_cols)), z_feature_cols, rotation=90)
plt.yticks(range(len(z_feature_cols)), z_feature_cols)
plt.title("Surface Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 21. Practical checklist for volatility surface alpha

Before calling a surface feature alpha, check:

1. **Surface construction**  
   Are quotes clean, liquid, and arbitrage-aware?

2. **No look-ahead**  
   Are realised-volatility features backward-looking only?

3. **Forward alignment**  
   Are targets properly shifted?

4. **Overlapping labels**  
   Are forward windows overlapping? If yes, account for dependence.

5. **Baselines**  
   Does the signal beat ATM IV, past RV, and naïve forecasts?

6. **Costs**  
   Does it survive option bid/ask, hedging, slippage, and margin?

7. **Regime dependence**  
   Does it fail in stress?

8. **Signal stability**  
   Is IC stable through time?

9. **Feature redundancy**  
   Is the signal just repackaged ATM volatility?

10. **Economic mechanism**  
   Is there a plausible explanation such as volatility risk premium, crash insurance demand, or positioning?

11. **Capacity**  
   Can the trade be sized in real options markets?

12. **Risk**  
   Does the signal hide short-tail exposure?

## 22. Saving outputs

The notebook saves:

1. synthetic market data;
2. synthetic surface data;
3. extracted surface features;
4. research feature/target table;
5. IC tables;
6. bucket diagnostics;
7. forecast metrics;
8. signal ICs;
9. toy PnL reports;
10. cost sensitivity;
11. regime diagnostics;
12. rolling IC;
13. manifest.

In [ ]:
output_dir = Path("data/processed/volatility_surface_alpha_signals_v1")
output_dir.mkdir(parents=True, exist_ok=True)

market_path = output_dir / "synthetic_market.csv"
surface_path = output_dir / "synthetic_vol_surface.csv"
features_path = output_dir / "surface_features.csv"
research_path = output_dir / "research_table.csv"
ic_train_path = output_dir / "ic_train.csv"
ic_test_path = output_dir / "ic_test.csv"
bucket_vrp_path = output_dir / "bucket_vrp.csv"
forecast_df_path = output_dir / "forecast_df.csv"
forecast_metrics_path = output_dir / "forecast_metrics.csv"
signal_ic_path = output_dir / "signal_ic.csv"
bucket_composite_path = output_dir / "bucket_composite.csv"
toy_summary_path = output_dir / "toy_pnl_summary.csv"
toy_pnl_path = output_dir / "toy_pnl.csv"
cost_sensitivity_path = output_dir / "cost_sensitivity.csv"
regime_pnl_path = output_dir / "regime_pnl.csv"
rolling_ic_path = output_dir / "rolling_ic.csv"
feature_corr_path = output_dir / "feature_correlation.csv"
manifest_path = output_dir / "manifest.json"

market.to_csv(market_path, index=False)
surface.to_csv(surface_path, index=False)
features.to_csv(features_path, index=False)
research_df.to_csv(research_path, index=False)
ic_train.to_csv(ic_train_path, index=False)
ic_test.to_csv(ic_test_path, index=False)
bucket_vrp.to_csv(bucket_vrp_path, index=False)
forecast_df.to_csv(forecast_df_path, index=False)
forecast_metrics.to_csv(forecast_metrics_path, index=False)
signal_ic.to_csv(signal_ic_path, index=False)
bucket_composite.to_csv(bucket_composite_path, index=False)
toy_summary.to_csv(toy_summary_path, index=False)
toy_pnl.to_csv(toy_pnl_path, index=False)
cost_sensitivity.to_csv(cost_sensitivity_path, index=False)
regime_pnl.to_csv(regime_pnl_path, index=False)
rolling_ic_df.to_csv(rolling_ic_path, index=False)
feature_corr.to_csv(feature_corr_path)

manifest = {
    "dataset_name": "volatility_surface_alpha_signals_outputs",
    "schema_version": "volatility_surface_alpha_signals_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "surface_signal_cols": surface_signal_cols,
    "z_feature_cols": z_feature_cols,
    "model_feature_cols": model_feature_cols,
    "signal_cols": signal_cols,
    "best_test_ic": ic_test.iloc[0].to_dict(),
    "best_toy_signal": toy_summary.iloc[0].to_dict(),
    "best_signal_name": str(surface_pnl_best_signal),
    "notes": [
        "Synthetic surface data is used for a controlled alpha-research workflow.",
        "Features include ATM IV, skew, curvature, term structure, wing spread, and VRP proxies.",
        "VRP is approximated using ATM IV squared minus past realised volatility squared.",
        "Signals are tested on a chronological test split.",
        "Toy PnL is a proxy for variance-risk-premium exposure, not a real option backtest.",
        "The notebook explicitly avoids claiming tradable alpha without costs, hedging, liquidity, and robustness tests."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

market_path, surface_path, research_path, toy_summary_path, manifest_path

## 23. Limitations

### 23.1 Synthetic surface

This notebook uses synthetic data.

Real option surfaces require quote filtering, bid/ask handling, expiry calendars, forwards, dividends, rates, and arbitrage checks.

### 23.2 ATM IV is not a variance swap

The VRP proxy uses ATM IV squared.

A true variance risk premium should use variance swap replication or model-free implied variance.

### 23.3 Toy PnL is not an option strategy

The toy PnL does not model:

- option premium;
- Greeks;
- delta hedging;
- gamma/theta;
- transaction costs;
- margin;
- early exercise;
- smile dynamics;
- liquidity.

### 23.4 Overlapping labels

Forward 21-day labels overlap, creating serial dependence.

Formal statistical tests should adjust for overlapping horizons.

### 23.5 Feature mining

Many surface features increase the chance of false discovery.

A production pipeline should include multiple-testing controls or strong holdout discipline.

### 23.6 Regime dependence

Volatility carry signals can lose sharply during stress.

Average performance can hide tail risk.

### 23.7 Single asset

The notebook uses one synthetic underlying.

Cross-sectional surface alpha would require many assets or futures options.

## 24. Research frontier and extensions

Important extensions include:

1. **Model-free implied variance**  
   Build variance swap replication from OTM option strips.

2. **SVI/SSVI surface features**  
   Extract arbitrage-aware parameters and use them as signals.

3. **SABR/Heston parameter alpha**  
   Use calibrated parameters such as $\rho,\nu,\theta,\xi$ as features.

4. **VRP term structure**  
   Forecast returns or realised volatility using variance premium across maturities.

5. **Skew risk premium**  
   Study whether downside protection demand predicts future returns or crash risk.

6. **Delta-hedged option PnL attribution**  
   Test whether signals predict actual option hedging PnL.

7. **Cross-sectional option surface signals**  
   Rank assets by VRP, skew, curvature, and term structure.

8. **Regime-conditioned surface alpha**  
   Combine HMM/GARCH regimes with volatility surface features.

9. **Chinese futures options extension**  
   Use Black-76 surfaces, commodity seasonality, night sessions, and contract-specific liquidity.

10. **Risk-controlled short-vol portfolio**  
   Add drawdown stops, convexity hedges, and stress overlays.

## 25. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_10_information_coefficient_analysis**  
   General signal IC framework for alpha features.

2. **03_11_tree_models_for_return_prediction**  
   Nonlinear models using surface features.

3. **04_06_var_cvar_and_stress_testing**  
   Stress-test surface signals.

4. **05_01_vectorized_backtest_engine**  
   Backtest signals with cost assumptions.

5. **05_04_option_strategy_backtester_delta_hedged**  
   Convert surface signals into option strategy PnL.

6. **07_02_volatility_surface_pricing_and_alpha**  
   Capstone combining surface construction, calibration, and alpha research.

## 26. Summary

This notebook implemented volatility surface alpha signal research.

It showed how to:

1. simulate daily implied volatility surfaces;
2. extract ATM IV, skew, curvature, wing spread, and term-structure features;
3. compute realised volatility and VRP proxies;
4. create rolling z-score signals;
5. align forward targets without look-ahead;
6. compute information coefficients;
7. run bucket diagnostics;
8. fit simple linear volatility forecasts;
9. define candidate surface alpha signals;
10. evaluate toy PnL proxies;
11. test cost sensitivity;
12. diagnose regime dependence;
13. check rolling IC stability;
14. save outputs and manifests.

The key computational takeaway:

> Surface alpha research is feature engineering plus strict validation, not just plotting a smile.

The key financial takeaway:

> Volatility surface signals may contain information about volatility risk premium and future risk, but any apparent edge must survive costs, stress regimes, and real option implementation.

## 27. Further reading

- Gatheral, *The Volatility Surface.*
- Carr and Madan on option-implied variance and Fourier methods.
- Bakshi, Kapadia, and Madan on risk-neutral moments.
- Bollerslev, Tauchen, and Zhou on volatility risk premium.
- Literature on variance risk premium, skew risk premium, option-implied moments, and delta-hedged option returns.